## 갤럽. 정기(정례)조사,정당지지도,

In [34]:
# 비율 추론. phat, se, 신뢰구간, 검정통계량, pvalue.

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest


### 긍정(부정) 평가 결과 일부 재현

In [35]:
# 여론조사 결과. 2026. 9. 1. 중앙선관위 게시판 #17264
# 정기(정례)조사,정당지지도, (전국 정기(정례)조사 정당지지도 )
# 조사기관 자체 : 한국갤럽 자체 조사

# 문 1) ○○님께서는 요즘 이재명 대통령이
# 대통령으로서의 직무를 잘 수행하고 있다고 보십니까,
# 아니면 잘못 수행하고 있다고 보십니까?
# (3, 9인 경우: “굳이 말씀하신다면
# ‘잘하고 있다’와 ‘잘못하고 있다’ 중 어느 쪽입니까?”)

# 원자료가 없고, 추정치만 있어서 그 값을 사용함.
# 가령
# 38%는 37.5~38.5% 사이의 값일 수 있음 375명~384명 구간.
# 51%는 50.5~51.5% 사이의 값일 수 있음 505명~514명 구간.

n = 1000
p_yay_hat = 0.375
p_nay_hat = 0.51



### 비율, 신뢰구간

In [49]:
# 리포트 옵션 1을 사용하는 듯.

p_hat = p_nay_hat   # 긍정 p_yay, 부정 p_nay
print("표본비율 ")
print( p_hat )

# 옵션 1, 옵션 2,
se_1 = np.sqrt( p_hat * (1 - p_hat) / n )
se_2 = np.sqrt( 0.5 * (1 - 0.5) / n )
print("표준오차,   옵션1,   옵션2(최대) ")
print(se_1, se_2)

print("상대표준오차, 옵션1,    옵션2 ")   # 표준오차/평균,
print( se_1 /p_hat, se_2 /p_hat )

alpha = 0.05                            # 유의수준, 임계치
z_c = abs( stats.norm.ppf( alpha/2 ) )
# z_2 = stats.norm.ppf( 1- alpha/2 )

ci_1 = p_hat - z_c * se_1  # 옵션1
ci_2 = p_hat + z_c * se_1

print(" 신뢰구간 ")
print(ci_1, ci_2 )

ci_1_max = p_hat - z_2 * se_2  # 옵션2
ci_2_max = p_hat + z_2 * se_2

print(" 최대 신뢰구간 ")
print(ci_1_max, ci_2_max )

표본비율 
0.51
표준오차,   옵션1,   옵션2(최대) 
0.0158082257068907 0.015811388300841896
상대표준오차, 옵션1,    옵션2 
0.030996520993903334 0.03100272215851352
 신뢰구간 
0.479016446955014 0.540983553044986
 최대 신뢰구간 
0.4790102483847719 0.5409897516152281


### 가설 검정  $ H_0 : p  $=0.5

In [46]:
p_zero = 0.5
se_3 = ( p_zero *( 1 - p_zero ) / n )**0.5
t_0 = abs( ( p_hat - p_zero ) / se_3  )
print("검정통계량 ")
print(t_0)
print(f"{t_0 } > {z_c} 이면 H0:p={p_zero} 기각. 유의수준={alpha}")

검정통계량 
0.6324555320336764
0.6324555320336764 > 1.9599639845400545 이면 H0:p=0.5 기각. 유의수준=0.05


### $ p $ value

In [48]:
p_cmlt = stats.norm.cdf(t_0, loc=0, scale=1)
p_val = 2 * (1 - p_cmlt)
print("p value ")
print(p_val)
print(f"{p_val} < {alpha} 이면 H0:p={p_zero} 기각.")

p value 
0.5270892568655376
0.5270892568655376 < 0.05 이면 H0:p=0.5 기각.
